# Day 18 — LSTM Sentiment Classifier

A bidirectional LSTM for binary sentiment classification, built in PyTorch.

This notebook walks through the full pipeline: data -> vocab -> model -> train -> predict.
Runs end-to-end offline using the synthetic-but-realistic review generator in `src/dataset.py`.
Flip `USE_REAL_IMDB = True` there to swap in the real IMDB dataset (one-line change).

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..", "src"))

import torch
from dataset import generate_synthetic_dataset, train_val_split, Vocab, SentimentDataset, collate_batch
from model import LSTMSentimentClassifier

torch.manual_seed(42)

## 1. Data

In [ ]:
raw = generate_synthetic_dataset(n_samples=6000, seed=42)
train_samples, val_samples = train_val_split(raw, val_frac=0.15, seed=42)
print(f"Train: {len(train_samples)}  Val: {len(val_samples)}")
print("Example:", raw[0])

## 2. Vocabulary + Dataset

In [ ]:
vocab = Vocab([t for t, _ in train_samples])
print(f"Vocab size: {len(vocab)}")

train_ds = SentimentDataset(train_samples, vocab, max_len=200)
val_ds = SentimentDataset(val_samples, vocab, max_len=200)

## 3. Model

In [ ]:
model = LSTMSentimentClassifier(vocab_size=len(vocab), embed_dim=128, hidden_dim=128, num_layers=2)
print(model)
n_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {n_params:,}")

## 4. Train

For a full run, use the CLI script instead (it saves checkpoints and a history.json):

```bash
python src/train.py --epochs 8 --batch-size 64
```

Below is a short in-notebook training loop for a few epochs, just to see the mechanics.

In [ ]:
from torch.utils.data import DataLoader
import torch.nn as nn

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collate_batch)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, collate_fn=collate_batch)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

for epoch in range(3):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for input_ids, lengths, labels in train_loader:
        input_ids, lengths, labels = input_ids.to(device), lengths.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(input_ids, lengths)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        total_loss += loss.item() * labels.size(0)
        preds = (torch.sigmoid(logits) >= 0.5).float()
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    print(f"Epoch {epoch+1}: loss {total_loss/total:.4f}  acc {correct/total:.4f}")

## 5. Predict on new text

In [ ]:
def predict(text, model, vocab, max_len=200):
    model.eval()
    ids = vocab.encode(text)[:max_len] or [vocab.stoi["<unk>"]]
    input_ids = torch.tensor([ids], dtype=torch.long, device=device)
    lengths = torch.tensor([len(ids)], dtype=torch.long)
    with torch.no_grad():
        prob = torch.sigmoid(model(input_ids, lengths)).item()
    label = "positive" if prob >= 0.5 else "negative"
    conf = prob if prob >= 0.5 else 1 - prob
    return label, conf

for text in [
    "what a fantastic and touching film, I loved it",
    "the plot was dull and the acting was wooden",
]:
    label, conf = predict(text, model, vocab)
    print(f"{text!r} -> {label} ({conf:.1%})")

## Notes

- This notebook trains for only 3 epochs on a small sample for demonstration speed — for real results use `src/train.py` with more epochs and samples.
- The synthetic dataset is a stand-in so the whole project runs offline; swap in real IMDB via `USE_REAL_IMDB = True` in `src/dataset.py` for meaningful accuracy numbers.